In [ ]:
!pip install conllu
!git clone https://github.com/UniversalDependencies/UD_Spanish-AnCora.git

In [13]:
import numpy as np
transition_prob_dict = np.load("/content/transitionHMM.npy", allow_pickle=True).item()
emission_prob_dict = np.load("/content/emissionHMM.npy", allow_pickle=True).item()

In [ ]:
emission_prob_dict

# Identificar las etiquetas gramaticales únicas

In [ ]:
state_set = set([w.split("|")[1] for w in list(emission_prob_dict.keys())])
state_set

In [29]:
len(state_set)

17

## Enumerar las etiquetas gramaticales

In [35]:
tag_state_dict = {}
for index, state in enumerate(state_set):
  tag_state_dict[state] = index

tag_state_dict

{'ADV': 0,
 'DET': 1,
 'CCONJ': 2,
 'AUX': 3,
 'PUNCT': 4,
 'VERB': 5,
 'SYM': 6,
 'SCONJ': 7,
 '_': 8,
 'PRON': 9,
 'ADJ': 10,
 'INTJ': 11,
 'PROPN': 12,
 'NUM': 13,
 'ADP': 14,
 'PART': 15,
 'NOUN': 16}

## Distribución inicial de estados

¿Cual es la probabilidad de que las oraciones del corpus empiecen con X etiqueta?

In [ ]:
from conllu import parse_incr

data_file = open("/content/UD_Spanish-AnCora/es_ancora-ud-dev.conllu", "r", encoding="utf-8")

init_tag_state_prob = {}
count = 0

for tokenlist in parse_incr(data_file):
  count+=1
  tag = tokenlist[0]['upos']
  if tag in init_tag_state_prob.keys():
    init_tag_state_prob[tag] += 1
  else:
    init_tag_state_prob[tag] = 1

for key in init_tag_state_prob.keys():
  init_tag_state_prob[key] = init_tag_state_prob[key]/count

init_tag_state_prob

In [24]:
np.array([init_tag_state_prob[k] for k in init_tag_state_prob.keys()]).sum()

1.0

# Algoritmo de Viterbi

In [32]:
import nltk
nltk.download("punkt_tab")
from nltk import word_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [58]:
def viterbi_matrix(texto,transition_prob_dict=transition_prob_dict,
                   emission_prob_dict=emission_prob_dict,
                   tag_state_dict=tag_state_dict, init_tag_state_prob=init_tag_state_prob):
  seq = word_tokenize(texto)
  viterbi_prob = np.zeros((17,len(seq)))
  # Primera fase: Inicialización de la primera columna (palabra)
  for key in tag_state_dict.keys():
    tag_row = tag_state_dict[key]
    word_tag = seq[0].lower() + "|" + key
    if word_tag in emission_prob_dict.keys():
      viterbi_prob[tag_row, 0] = init_tag_state_prob[key] * emission_prob_dict[word_tag]
  # Segunda fase: Procesamiento de las columnas restantes
  for col in range(1,len(seq)):
    for key in tag_state_dict.keys():
      tag_row = tag_state_dict[key]
      word_tag = seq[col].lower() + "|" + key
      if word_tag in emission_prob_dict.keys(): # Considera todas las posibles etiquetas de la palabra anterior
        possible_probs=[]
        for key2 in tag_state_dict.keys():
          tag_row2 = tag_state_dict[key2]
          tag_prevtag = key + "|" + key2
          if tag_prevtag in transition_prob_dict.keys():
            if viterbi_prob[tag_row2, col-1] > 0:
              possible_probs.append(viterbi_prob[tag_row2, col-1] * transition_prob_dict[tag_prevtag] * emission_prob_dict[word_tag])
        viterbi_prob[tag_row, col] = max(possible_probs)
  return viterbi_prob

matrix = viterbi_matrix("el gato negro")
matrix

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.24339097e-01, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 4.39901860e-10],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 9.13941530e-06, 0.00000000e+00]])

In [67]:
def viterbi_tagger(texto,transition_prob_dict=transition_prob_dict,
                   emission_prob_dict=emission_prob_dict,
                   tag_state_dict=tag_state_dict, init_tag_state_prob=init_tag_state_prob):
  seq = word_tokenize(texto)
  viterbi_prob = np.zeros((17,len(seq)))
  # Primera fase: Inicialización de la primera columna (palabra)
  for key in tag_state_dict.keys():
    tag_row = tag_state_dict[key]
    word_tag = seq[0].lower() + "|" + key
    if word_tag in emission_prob_dict.keys():
      viterbi_prob[tag_row, 0] = init_tag_state_prob[key] * emission_prob_dict[word_tag]
  # Segunda fase: Procesamiento de las columnas restantes
  for col in range(1,len(seq)):
    for key in tag_state_dict.keys():
      tag_row = tag_state_dict[key]
      word_tag = seq[col].lower() + "|" + key
      if word_tag in emission_prob_dict.keys(): # Considera todas las posibles etiquetas de la palabra anterior
        possible_probs=[]
        for key2 in tag_state_dict.keys():
          tag_row2 = tag_state_dict[key2]
          tag_prevtag = key + "|" + key2
          if tag_prevtag in transition_prob_dict.keys():
            if viterbi_prob[tag_row2, col-1] > 0:
              possible_probs.append(viterbi_prob[tag_row2, col-1] * transition_prob_dict[tag_prevtag] * emission_prob_dict[word_tag])
        viterbi_prob[tag_row, col] = max(possible_probs)
  # Tercera fase: Decodificar - Encontrar la mejor secuencia de etiquetas
  seq_tags = []
  for i, p in enumerate(seq):
    for tag in tag_state_dict.keys():
      if tag_state_dict[tag] == np.argmax(viterbi_prob[:,i]):
        seq_tags.append((p,tag))
  return seq_tags

viterbi_tagger("Encontrar la mejor secuencia de etiquetas")

[('Encontrar', 'VERB'),
 ('la', 'DET'),
 ('mejor', 'ADJ'),
 ('secuencia', 'NOUN'),
 ('de', 'ADP'),
 ('etiquetas', 'ADV')]